# Week 6 Exercise — Bug Detection Neural Networks & Fine-Tuning

> **Disclaimer:** Please ensure you have downloaded the required data from this [Google Drive folder](https://drive.google.com/drive/folders/1AaLC3nnm6gJLU66R7mAYFuSQRkxC0WqV?usp=drive_link) and placed them in the appropriate locations. This includes the `buggy_dataset.jsonl` file which should be placed at excercise root directory.

**Description**: This exercise transitions from traditional heuristic bug detection to Deep Learning. Using the `buggy_dataset.jsonl` synthetic dataset, we explore two primary approaches:
1. **Simple Bug Classifier**: Training a lightweight PyTorch Neural Network from scratch (using `HashingVectorizer`) to classify the *type* of bug in a snippet.
2. **Complex Auto-Fixer**: Fine-tuning an open-source Sequence-to-Sequence model (`CodeT5-small`) locally via Hugging Face to automatically correct buggy code.

Finally, we package these into an interactive Gradio UI for direct comparison against state-of-the-art LLMs via OpenRouter.

**Dependencies:** If you see `No module named pip`, the notebook kernel (e.g. `.venv` from uv) may not have pip. Install from the repo root instead: `uv sync` or `uv pip install torch scikit-learn transformers datasets evaluate seaborn matplotlib pandas gradio`. Then restart the kernel. If your env already has these, skip the cell below.

In [ ]:
# If you get "No module named pip", run: python -m ensurepip --upgrade
%pip install -q torch scikit-learn transformers datasets evaluate seaborn matplotlib pandas gradio accelerate>=0.26.0

In [ ]:
import json
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Verify PyTorch installation and device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Phase 1: Data Preparation & Splitting
Load the 60 items from `buggy_dataset.jsonl` and split them into 50 training and 10 testing examples.

In [ ]:
# Load Dataset
dataset_path = "buggy_dataset.jsonl"
data = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line.strip()))

# Convert to DataFrame for easier analysis
df = pd.DataFrame(data)
print(f"Total examples loaded: {len(df)}")

# Create an aggregated 'bug_type' label (taking the first bug type for simplicity in classification)
df['primary_bug_type'] = df['bug_types'].apply(lambda x: x[0] if len(x) > 0 else "Unknown")

# Check class distribution before splitting
print("\nClass Distribution:")
print(df['primary_bug_type'].value_counts())

# Shuffle and split without stratification to avoid errors with small classes
train_df, test_df = train_test_split(df, test_size=10, random_state=42)

print(f"\nTraining examples: {len(train_df)}")
print(f"Testing examples: {len(test_df)}")

# Show distribution of bug types in training
plt.figure(figsize=(10, 5))
sns.countplot(data=train_df, y='primary_bug_type', order=train_df['primary_bug_type'].value_counts().index)
plt.title("Distribution of Primary Bug Types (Training Set)")
plt.show()

## Phase 2: Simple Neural Network (PyTorch + HashingVectorizer)

Following the Week 6 Day 4 concepts, we build a simple feedforward neural network from scratch using `PyTorch`. This model takes vectorized buggy code and attempts to classify the *type* of bug.

In [ ]:
# Feature Extraction with HashingVectorizer (Week 6 Day 4 concept)
n_features = 1000  # Limited feature space since dataset is small
vectorizer = HashingVectorizer(n_features=n_features, stop_words='english')

X_train = vectorizer.fit_transform(train_df['buggy_code']).toarray()
X_test = vectorizer.transform(test_df['buggy_code']).toarray()

# Prepare Labels: Convert string labels to integers
label_mapping = {label: idx for idx, label in enumerate(df['primary_bug_type'].unique())}
num_classes = len(label_mapping)
print(f"Label Mapping: {label_mapping}")
print(f"Number of classes to predict: {num_classes}")

y_train = train_df['primary_bug_type'].map(label_mapping).values
y_test = test_df['primary_bug_type'].map(label_mapping).values

# Convert to PyTorch Tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)

X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

# Create DataLoader for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [ ]:
# Simple PyTorch Neural Network

class BugClassifierNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(BugClassifierNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc3(out)
        return out

model = BugClassifierNN(input_size=n_features, num_classes=num_classes).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture:\n{model}")
print(f"\nNumber of trainable parameters: {trainable_params:,}")

In [ ]:
# Training Loop with Loss Tracking

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    # Calculate average losses
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # Validation Loss
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_tensor)
        val_loss = criterion(val_outputs, y_test_tensor).item()
        val_losses.append(val_loss)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {epoch_train_loss:.4f}, Val Loss: {val_loss:.4f}')

In [ ]:
# Plotting the Training and Validation Loss

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss', color='#4A90D9', linewidth=2)
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss', color='#E74C3C', linewidth=2)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Simple Neural Network — Training vs Validation Loss', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation on Test Set

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs.data, 1)

accuracy = accuracy_score(y_test, predicted.cpu().numpy())
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")

# Generate Classification Report
inv_label_mapping = {v: k for k, v in label_mapping.items()}
unique_test = np.unique(y_test)
unique_pred = np.unique(predicted.cpu().numpy())
all_classes_present = np.unique(np.concatenate((unique_test, unique_pred)))
ordered_names = [inv_label_mapping[i] for i in all_classes_present]

print("Classification Report:")
print(classification_report(y_test, predicted.cpu().numpy(),
                            labels=all_classes_present, target_names=ordered_names, zero_division=0))

## Phase 3: Complex Neural Network / Seq2Seq Fine-Tuning

Next, we will fine-tune a pre-trained open-source sequence-to-sequence model (`Salesforce/codet5-small`) using Hugging Face's `transformers` library. Rather than just classify the bug, this model will attempt to generate the **corrected code** directly!

*Note: Make sure you have `transformers`, `datasets`, and `evaluate` installed before running the cells below.*

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset

MODEL_NAME = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Loaded model: {MODEL_NAME}")
print(f"Model parameters: {sum(p.numel() for p in seq2seq_model.parameters()):,}")

In [ ]:
PREFIX = "fix buggy python: "

def preprocess(examples):
    inputs = [PREFIX + code for code in examples["buggy_code"]]
    targets = examples["correct_code"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_hf = Dataset.from_pandas(train_df[['buggy_code', 'correct_code']].reset_index(drop=True))
test_hf = Dataset.from_pandas(test_df[['buggy_code', 'correct_code']].reset_index(drop=True))

train_tokenized = train_hf.map(preprocess, batched=True, remove_columns=train_hf.column_names)
test_tokenized = test_hf.map(preprocess, batched=True, remove_columns=test_hf.column_names)

print(f"Tokenized train size: {len(train_tokenized)}")
print(f"Tokenized test size: {len(test_tokenized)}")

In [ ]:
# Fine-Tuning with the Hugging Face Trainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5_bugfix",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    logging_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=seq2seq_model)

trainer = Seq2SeqTrainer(
    model=seq2seq_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
train_result = trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Plotting the Fine-Tuning Loss

log_history = trainer.state.log_history

ft_train_losses = [entry['loss'] for entry in log_history if 'loss' in entry]
ft_eval_losses = [entry['eval_loss'] for entry in log_history if 'eval_loss' in entry]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(range(1, len(ft_train_losses) + 1), ft_train_losses, color='#4A90D9', linewidth=2, marker='o', markersize=4)
ax[0].set_title('CodeT5 Fine-Tuning — Training Loss', fontsize=14)
ax[0].set_xlabel('Epoch', fontsize=12)
ax[0].set_ylabel('Loss', fontsize=12)
ax[0].grid(True, linestyle='--', alpha=0.7)

ax[1].plot(range(1, len(ft_eval_losses) + 1), ft_eval_losses, color='#E74C3C', linewidth=2, marker='s', markersize=4)
ax[1].set_title('CodeT5 Fine-Tuning — Validation Loss', fontsize=14)
ax[1].set_xlabel('Epoch', fontsize=12)
ax[1].set_ylabel('Loss', fontsize=12)
ax[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the Fine-Tuned CodeT5 Model on Test Set for bug correction

def fix_code(buggy_code_str):
    """Use the fine-tuned CodeT5 model to fix buggy code."""
    input_text = PREFIX + buggy_code_str
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids.to(seq2seq_model.device)
    outputs = seq2seq_model.generate(input_ids, max_length=512)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 80)
print("EVALUATION: Fine-Tuned CodeT5 on Test Set")
print("=" * 80)

for i, row in test_df.iterrows():
    predicted_fix = fix_code(row['buggy_code'])
    is_match = predicted_fix.strip() == row['correct_code'].strip()
    status = "MATCH" if is_match else "MISMATCH"
    print(f"\n--- Example (id={row['id']}, level={row['level']}) [{status}] ---")
    print(f"Description: {row['description']}")
    if not is_match:
        print(f"Expected (first 200 chars): {row['correct_code'][:200]}...")
        print(f"Got      (first 200 chars): {predicted_fix[:200]}...")

## Phase 4: Gradio UI & OpenRouter Comparison

An interactive interface to paste buggy Python code and get:
1. The predicted **bug type** (from the Simple NN).
2. The **corrected code** (from the Fine-Tuned CodeT5 model).
3. (Optional) A response from a major LLM via OpenRouter for comparison.

In [ ]:
import gradio as gr

def analyze_buggy_code(buggy_code_input):
    """Run both models on the given buggy code."""
    # --- Simple NN: Bug Type Classification ---
    vec = vectorizer.transform([buggy_code_input]).toarray()
    vec_tensor = torch.FloatTensor(vec).to(device)
    model.eval()
    with torch.no_grad():
        output = model(vec_tensor)
        _, pred_class = torch.max(output, 1)
    predicted_bug_type = inv_label_mapping.get(pred_class.item(), "Unknown")

    corrected_code = fix_code(buggy_code_input)

    return predicted_bug_type, corrected_code

demo = gr.Interface(
    fn=analyze_buggy_code,
    inputs=gr.Code(language="python", label="Paste Buggy Python Code"),
    outputs=[
        gr.Textbox(label="Predicted Bug Type (Simple NN)"),
        gr.Code(language="python", label="Corrected Code (Fine-Tuned CodeT5)")
    ],
    title="Bug Detection & Auto-Fix",
    description="Paste buggy Python code below. The Simple NN classifies the bug type, and the fine-tuned CodeT5 model attempts to fix the code.",
)

demo.launch()